In [1]:
from document_extractor import extract_itac_report

doc_1_path = "/Users/afschowdhury/Code Local/itac-report-validator/docs/report1/LS2502 - Final Draft R2.docx"
doc_2_path = "/Users/afschowdhury/Code Local/itac-report-validator/docs/report2/LS2508 - Final Draft.docx"
out = extract_itac_report(doc_2_path, output="html", save_files=True)


In [2]:
out.keys()


dict_keys(['general_information', 'annual_energy_usages_and_costs', 'carbon_footprint', 'recommendation_summary_table', 'ar_summary', 'assessment_recommendations'])

In [3]:
recommendations = out['assessment_recommendations']


In [4]:
from IPython.display import HTML
HTML(recommendations[0])


In [5]:
# Test the new get_single_ar_summary_table function
from doc_extractor_utils import get_single_ar_summary_table

# Extract data from the first AR
ar_1_data = get_single_ar_summary_table(recommendations[0])
print(f"AR Number: {ar_1_data['ar_number']} (type: {type(ar_1_data['ar_number'])})")
print(f"\nHeaders: {ar_1_data['headers']}")
print(f"\nData: {ar_1_data['data']}")


AR Number: 1 (type: <class 'int'>)

Headers: ['Energy Savings (kWh/yr)', 'Energy Cost Savings($/yr)', 'Demand Savings (kW/yr)', 'Demand Cost($/yr)', 'Total Cost Savings ($/yr)', 'CO2Reduction (tons/yr)', 'Imp. Cost ($)', 'Payback Period(yrs)']

Data: {'electricity_savings_kwh_per_year': 6806, 'energy_cost_savingsdollar_per_yr': 694, 'demand_savings_kw_per_year': 17, 'demand_costdollar_per_yr': 77, 'total_cost_savings_per_year': 771, 'co2reduction_tons_per_yr': 3, 'implementation_cost': 675, 'payback_period_years': 0.88}


In [11]:
ar_1_data['data']

{'electricity_savings_kwh_per_year': 6806,
 'energy_cost_savingsdollar_per_yr': 694,
 'demand_savings_kw_per_year': 17,
 'demand_costdollar_per_yr': 77,
 'total_cost_savings_per_year': 771,
 'co2reduction_tons_per_yr': 3,
 'implementation_cost': 675,
 'payback_period_years': 0.88}

In [6]:
# Now let's get the recommendation summary table
from doc_extractor_utils import get_recommended_summary_table_json

rec_summary = get_recommended_summary_table_json(out['recommendation_summary_table'])

# Show the recommendations with AR numbers as integers
print("Recommendations from summary table:")
for rec in rec_summary['recommendations']:
    print(f"  AR {rec['ar_number']} (type: {type(rec['ar_number'])}): {rec.get('description', 'N/A')}")


Recommendations from summary table:
  AR 1 (type: <class 'int'>): Utilize Higher Efficiency Lamps and/or Ballasts
  AR 2 (type: <class 'int'>): Install Sub-metering Equipment
  AR 3 (type: <class 'int'>): Modify Inventory Control
  AR 4 (type: <class 'int'>): Replace Existing HVAC with Higher Efficiency Model
  AR 5 (type: <class 'int'>): Use Solar Heat to Generate Electricity
  AR 6 (type: <class 'int'>): Consider Replacement of Old Motors with Energy-Efficient Ones
  AR 7 (type: <class 'int'>): Purchase Optimum Sized Air Compressor with More Suitable Substitutes
  AR 8 (type: <class 'int'>): Replace Fossil Fuel Equipment with Electrical Equipment


In [7]:
# Test the updated compare_ar_with_summary function
from doc_extractor_utils import compare_ar_with_summary

# Compare AR 1 - now the function automatically finds the matching recommendation
comparison = compare_ar_with_summary(ar_1_data, rec_summary['recommendations'])

print(f"Comparing AR Number: {comparison.get('ar_number')}")
print(f"Total matches: {comparison['total_matches']}")
print(f"Total differences: {comparison['total_differences']}")

if comparison.get('error'):
    print(f"\nError: {comparison['error']}")
elif comparison['differences']:
    print("\nDifferences found:")
    for diff in comparison['differences']:
        print(f"  {diff['field']}: AR={diff['ar_value']} vs Summary={diff['summary_value']} (diff={diff.get('difference')})")
else:
    print("\nAll values match!")

if comparison.get('matches'):
    print(f"\nMatching fields ({len(comparison['matches'])}):")
    for match in comparison['matches'][:5]:  # Show first 5
        print(f"  {match['field']}: {match['ar_value']}")


Comparing AR Number: 1
Total matches: 5
Total differences: 0

All values match!

Matching fields (5):
  electricity_savings_kwh_per_year: 6806
  demand_savings_kw_per_year: 17
  total_cost_savings_per_year: 771
  implementation_cost: 675
  payback_period_years: 0.88


In [8]:
# Compare all ARs
print("Comparing all ARs with summary table:\n")

for i, ar_html in enumerate(recommendations):
    ar_data = get_single_ar_summary_table(ar_html)
    comparison = compare_ar_with_summary(ar_data, rec_summary['recommendations'])
    
    ar_num = comparison.get('ar_number', 'Unknown')
    status = "✓ MATCH" if comparison['total_differences'] == 0 else "✗ MISMATCH"
    
    print(f"AR {ar_num}: {status}")
    print(f"  Matches: {comparison['total_matches']}, Differences: {comparison['total_differences']}")
    
    if comparison.get('error'):
        print(f"  Error: {comparison['error']}")
    elif comparison['differences']:
        print("  Differences:")
        for diff in comparison['differences']:
            print(f"    - {diff['field']}: AR={diff['ar_value']} vs Summary={diff['summary_value']}")
    print()


Comparing all ARs with summary table:

AR 1: ✓ MATCH
  Matches: 5, Differences: 0

AR 2: ✓ MATCH
  Matches: 4, Differences: 0

AR 3: ✓ MATCH
  Matches: 6, Differences: 0

AR 4: ✓ MATCH
  Matches: 6, Differences: 0

AR 5: ✓ MATCH
  Matches: 2, Differences: 0

AR 6: ✓ MATCH
  Matches: 6, Differences: 0

AR 7: ✓ MATCH
  Matches: 6, Differences: 0

AR 8: ✓ MATCH
  Matches: 4, Differences: 0



In [9]:
# More detailed comparison for a specific AR
ar_number_to_check = 1

ar_data = get_single_ar_summary_table(recommendations[ar_number_to_check - 1])
comparison = compare_ar_with_summary(ar_data, rec_summary['recommendations'])

print(f"Detailed Comparison for AR {ar_number_to_check}")
print("=" * 60)
print(f"\nAR Data from individual Savings Summary table:")
for field, value in ar_data['data'].items():
    print(f"  {field}: {value}")

print(f"\nSummary Table Data for AR {ar_number_to_check}:")
matching_rec = next(
    (rec for rec in rec_summary['recommendations'] if rec.get('ar_number') == ar_number_to_check),
    None
)
if matching_rec:
    for field, value in matching_rec.items():
        if field in ar_data['data']:
            print(f"  {field}: {value}")

print(f"\nComparison Results:")
print(f"  Total matches: {comparison['total_matches']}")
print(f"  Total differences: {comparison['total_differences']}")


Detailed Comparison for AR 1

AR Data from individual Savings Summary table:
  electricity_savings_kwh_per_year: 6806
  energy_cost_savingsdollar_per_yr: 694
  demand_savings_kw_per_year: 17
  demand_costdollar_per_yr: 77
  total_cost_savings_per_year: 771
  co2reduction_tons_per_yr: 3
  implementation_cost: 675
  payback_period_years: 0.88

Summary Table Data for AR 1:
  electricity_savings_kwh_per_year: 6806
  demand_savings_kw_per_year: 17
  total_cost_savings_per_year: 771
  implementation_cost: 675
  payback_period_years: 0.88

Comparison Results:
  Total matches: 5
  Total differences: 0
